# VA-Regression v2 — Régression subject-specific de la Valence & de l'Arousal

**Objectif** : entraîner un modèle par sujet qui prédit `[Valence, Arousal]` à partir de l'EEG + signaux périphériques (GSR, BVP, Resp, EMG, Temp).

**Méthodologie**
- 14 canaux EEG × 5 bandes (Theta..Gamma) = 70 features bandpower
- 6 signaux périphériques → 15 features statistiques (mean/std/range/slope)
- Fenêtre 2s (256 samples), step 0.125s (16 samples), sampling 128 Hz
- Subject-specific : un modèle entraîné par sujet

**Changements vs v1 (corrections méthodologiques)**
1. **Baseline retirée** : les 3 premières secondes (pré-stimulus) de chaque essai DEAP ne sont plus fenêtrées — elles portaient à tort le label émotionnel de l'essai.
2. **Welch réellement moyenné** : `nperseg < len(signal)` → vraie réduction de variance de la PSD (au lieu d'un périodogramme brut).
3. **Window-level en GroupKFold (5 folds)** au lieu d'un unique `GroupShuffleSplit` → estimations stables.
4. **Nouveau cadrage honnête : Leave-One-Video-Out au niveau vidéo** — features agrégées par vidéo → 40 points/sujet, 40 folds. C'est la taille réelle du problème sur DEAP (1 label par vidéo).
5. Warnings non masqués globalement ; cache versionné (`cache_full_v2`).

## 1. Imports et configuration

> **Explication technique — imports**
>
> - `welch` (scipy) : estimation de densité spectrale de puissance (PSD) par la méthode de Welch (segments fenêtrés + moyenne) → sert à calculer la bandpower EEG.
> - `GroupKFold` / `LeaveOneGroupOut` : validations croisées qui gardent toutes les lignes d'un même *groupe* (ici une vidéo) dans le même fold → empêchent toute fuite entre fenêtres voisines très corrélées (94 % de recouvrement).
> - `StandardScaler` : centre/réduit les features ; **fitté uniquement sur le train de chaque fold** pour ne pas fuiter les stats du test.
> - `MultiOutputRegressor` : enveloppe un régresseur mono-sortie pour prédire les 2 cibles (V, A) ; RandomForest est nativement multi-sortie, GradientBoosting non.
> - `DummyRegressor` : baseline « prédit la moyenne » → ancre la lecture du R² (R²≈0 = aussi bon que la moyenne ; R²<0 = pire).
> - On **ne masque plus** les warnings globalement (v1 cachait la dépréciation de `np.trapz` et d'éventuels warnings numériques de `r2_score`).

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

# np.trapz est déprécié en numpy >= 2 ; np.trapezoid est le nouveau nom.
trapz = getattr(np, "trapezoid", np.trapz)

sns.set_style('whitegrid')

> **Explication technique — configuration**
>
> - `EEG_CHANNELS` : 14 indices de canaux (0-based) cohérents avec `test.ipynb`. `BANDS` : 5 bandes de fréquences classiques en EEG affectif.
> - `WINDOW_SIZE=256` (2 s @ 128 Hz), `STEP_SIZE=16` (0.125 s) → fenêtres glissantes à fort recouvrement (utile pour densifier, géré côté CV par groupes).
> - **`BASELINE_SAMPLES = 3 * 128 = 384`** : DEAP fournit 8064 échantillons/essai = **3 s de baseline pré-stimulus + 60 s de stimulus**. On commence le fenêtrage à l'échantillon 384 pour exclure la baseline (sinon ~24 fenêtres/essai sont étiquetées avec une émotion qui n'a pas encore été déclenchée).
> - **`WELCH_NPERSEG = 128`** (< 256) : Welch découpe alors la fenêtre en segments de 128 avec recouvrement 50 % et **moyenne ~3 périodogrammes** → PSD à variance réduite (en v1, `nperseg=len(signal)` ne moyennait rien).
> - `CACHE_DIR` versionné `cache_full_v2` : les features changent (baseline + Welch) donc l'ancien cache ne doit pas être réutilisé silencieusement.

In [2]:
DATA_PATH = r"E:\COURS ECOLE\COURS PGE 4\AI CLINIC\data\deap-dataset\data_preprocessed_python"
CACHE_DIR = r"E:\COURS ECOLE\COURS PGE 4\AI CLINIC\notebooks\cache_full_v2"
os.makedirs(CACHE_DIR, exist_ok=True)

EEG_CHANNELS = [1, 2, 3, 4, 6, 11, 13, 17, 19, 20, 21, 25, 29, 31]
BAND_NAMES = ['Theta', 'Alpha', 'LowBeta', 'HighBeta', 'Gamma']
BANDS = [(4, 8), (8, 12), (12, 16), (16, 25), (25, 45)]

SAMPLE_RATE   = 128
WINDOW_SIZE   = 256        # 2 s
STEP_SIZE     = 16         # 0.125 s
BASELINE_SEC  = 3
BASELINE_SAMPLES = BASELINE_SEC * SAMPLE_RATE   # 384 : pré-stimulus DEAP, exclu
WELCH_NPERSEG = 128        # < WINDOW_SIZE -> Welch moyenne vraiment les segments

# Canaux périphériques DEAP (indices 32..39, 0-based)
PERIPH = {
    'hEOG': 32, 'vEOG': 33,
    'zEMG': 34, 'tEMG': 35,
    'GSR':  36, 'Resp': 37,
    'BVP':  38, 'Temp': 39,
}

SUBJECT_LIST = [f'{i:02d}' for i in range(1, 33)]
print(f'{len(SUBJECT_LIST)} sujets — features : {len(EEG_CHANNELS)*len(BANDS)} EEG + 15 periph = '
      f'{len(EEG_CHANNELS)*len(BANDS) + 15}')
print(f'Baseline exclue : {BASELINE_SAMPLES} echantillons ({BASELINE_SEC}s) au debut de chaque essai')

32 sujets — features : 70 EEG + 15 periph = 85
Baseline exclue : 384 echantillons (3s) au debut de chaque essai


## 2. Extraction des features

Bandpowers EEG (Welch + intégration trapèze) + features statistiques sur les signaux périphériques. GSR et BVP sont particulièrement informatifs pour l'arousal (sudation, fréquence cardiaque).

> **Explication technique — `bandpower` & `peripheral_features`**
>
> **`bandpower`** : `welch` renvoie les fréquences et la PSD ; pour chaque bande on sélectionne les bins fréquentiels `lo<=f<=hi` et on **intègre la PSD au trapèze** (`trapz`) → puissance dans la bande (unité²/Hz × Hz). `nperseg=min(WELCH_NPERSEG, len)` garantit l'effet de moyennage de Welch même si la fenêtre est courte.
>
> **`peripheral_features`** : reçoit les 8 canaux périphériques (indices locaux 0..7 = canaux DEAP 32..39). Mapping vérifié : `2=zEMG, 3=tEMG, 4=GSR, 5=Resp, 6=BVP, 7=Temp`. On calcule 15 stats :
> - EMG (zEMG/tEMG) : `mean(|x|)` (énergie de contraction) + `std`.
> - GSR : `mean` (niveau tonique), `std`, `range` (composante phasique), `slope` (régression linéaire = tendance) → 4 features.
> - Resp : `mean`, `std`. BVP : `mean`, `std`, `range` (proxy amplitude pouls). Temp : `mean`, `slope`.
> - `np.polyfit(t, x, 1)[0]` = pente de la droite des moindres carrés (dérive lente du signal).
> - hEOG/vEOG (indices 0,1) volontairement ignorés (artefacts oculaires, non émotionnels).

In [3]:
def bandpower(signal, bands, sf=SAMPLE_RATE):
    """Puissance de `signal` dans chaque bande via Welch (PSD moyennee) + integration trapeze."""
    nperseg = min(WELCH_NPERSEG, len(signal))
    freqs, psd = welch(signal, sf, nperseg=nperseg)
    out = np.empty(len(bands))
    for i, (lo, hi) in enumerate(bands):
        idx = (freqs >= lo) & (freqs <= hi)
        out[i] = trapz(psd[idx], freqs[idx])
    return out


def peripheral_features(periph_window):
    """periph_window: (8, window_size) -> 15 features (zEMG,tEMG,GSR,Resp,BVP,Temp)."""
    # 0=hEOG 1=vEOG 2=zEMG 3=tEMG 4=GSR 5=Resp 6=BVP 7=Temp
    feats = []
    z = periph_window[2]
    feats += [np.mean(np.abs(z)), np.std(z)]                                  # zEMG
    t = periph_window[3]
    feats += [np.mean(np.abs(t)), np.std(t)]                                  # tEMG
    g = periph_window[4]
    feats += [np.mean(g), np.std(g), np.max(g) - np.min(g),
              np.polyfit(np.arange(len(g)), g, 1)[0]]                          # GSR
    r = periph_window[5]
    feats += [np.mean(r), np.std(r)]                                          # Resp
    b = periph_window[6]
    feats += [np.mean(b), np.std(b), np.max(b) - np.min(b)]                   # BVP
    tp = periph_window[7]
    feats += [np.mean(tp), np.polyfit(np.arange(len(tp)), tp, 1)[0]]          # Temp
    return np.array(feats)


PERIPH_FEATURE_NAMES = [
    'zEMG_mean', 'zEMG_std',
    'tEMG_mean', 'tEMG_std',
    'GSR_mean', 'GSR_std', 'GSR_range', 'GSR_slope',
    'Resp_mean', 'Resp_std',
    'BVP_mean', 'BVP_std', 'BVP_range',
    'Temp_mean', 'Temp_slope',
]
EEG_FEATURE_NAMES = [f'ch{ch}_{band}' for ch in EEG_CHANNELS for band in BAND_NAMES]
ALL_FEATURE_NAMES = EEG_FEATURE_NAMES + PERIPH_FEATURE_NAMES
print(f'Total features: {len(ALL_FEATURE_NAMES)}')

Total features: 85


> **Explication technique — `extract_subject_features`**
>
> - Charge `sX.dat` (pickle latin-1) : `data` = (40 essais, 40 canaux, 8064 échantillons), `labels` = (40, 4) dont les colonnes 0,1 = Valence, Arousal (échelle 1–9).
> - `eeg = data[trial,:32]`, `periph = data[trial,32:40]`, `va = labels[trial,:2]`.
> - **`start = BASELINE_SAMPLES`** : le fenêtrage démarre après les 3 s de baseline → seules les fenêtres du stimulus reçoivent le label émotionnel (correction clé v2).
> - Boucle `while start + WINDOW_SIZE <= n` : fenêtres de 256, pas de 16. Pour chaque fenêtre : 70 bandpowers EEG + 15 stats périph concaténés.
> - `video_ids` mémorise l'indice de l'essai (0..39) → sert de **groupe** pour la CV (toutes les fenêtres d'une vidéo restent ensemble).
> - Cache `.npz` compressé sous `cache_full_v2` ; `force=True` force le recalcul (utile car les features ont changé vs v1).

In [4]:
def extract_subject_features(subject_id, force=False):
    """Features (EEG+periph) d'un sujet, cache .npz. Returns X(n,85), y(n,2), video_ids(n,)."""
    cache_file = os.path.join(CACHE_DIR, f's{subject_id}_full_v2.npz')
    if os.path.exists(cache_file) and not force:
        d = np.load(cache_file)
        return d['X'], d['y'], d['video_ids']

    with open(os.path.join(DATA_PATH, f's{subject_id}.dat'), 'rb') as f:
        sub = pickle.load(f, encoding='latin1')
    data, labels = sub['data'], sub['labels']      # (40,40,8064), (40,4)

    X_list, y_list, vid_list = [], [], []
    for trial in range(40):
        eeg    = data[trial, :32]
        periph = data[trial, 32:40]
        va     = labels[trial, :2]                 # [Valence, Arousal]

        start = BASELINE_SAMPLES                   # <-- on saute la baseline pre-stimulus
        while start + WINDOW_SIZE <= eeg.shape[1]:
            eeg_feats = []
            for ch in EEG_CHANNELS:
                eeg_feats.extend(bandpower(eeg[ch, start:start+WINDOW_SIZE], BANDS))
            periph_feats = peripheral_features(periph[:, start:start+WINDOW_SIZE])
            X_list.append(np.concatenate([eeg_feats, periph_feats]))
            y_list.append(va)
            vid_list.append(trial)
            start += STEP_SIZE

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    vids = np.array(vid_list, dtype=np.int16)
    np.savez_compressed(cache_file, X=X, y=y, video_ids=vids)
    return X, y, vids

> **Explication technique — extraction batch**
>
> Boucle sur les 32 sujets : première passe ≈ 30 s/sujet (Welch coûteux), puis instantanée via cache. `force=True` ici pour **reconstruire le cache v2** (features différentes de v1). On affiche un exemple : avec la baseline retirée le nombre de fenêtres/sujet diminue légèrement vs v1 (24 fenêtres × 40 essais en moins).

In [6]:
print('Extraction v2 (baseline retiree + Welch moyenne) pour les 32 sujets...')
for sid in tqdm(SUBJECT_LIST):
    extract_subject_features(sid, force=True)

X_demo, y_demo, vids_demo = extract_subject_features('01')
print(f'\nSujet 01 -> X={X_demo.shape}, y={y_demo.shape}, vids uniques={len(np.unique(vids_demo))}')
print(f'Fenetres par video (sujet 01) : {len(vids_demo)//len(np.unique(vids_demo))}')

Extraction v2 (baseline retiree + Welch moyenne) pour les 32 sujets...


  0%|          | 0/32 [00:00<?, ?it/s]


Sujet 01 -> X=(18600, 85), y=(18600, 2), vids uniques=40
Fenetres par video (sujet 01) : 465


## 3. Régression window-level (GroupKFold, estimation stable)

On garde l'approche fenêtre par fenêtre de v1 mais en **5-fold GroupKFold** (groupes = vidéos) au lieu d'un seul split. On verra que le R² window-level reste structurellement négatif sur DEAP — ce n'est pas un bug, c'est la conséquence d'1 seul label par vidéo (voir §4).

> **Explication technique — `evaluate_window_and_video` & `train_subject_cv`**
>
> **`evaluate_window_and_video`** : métriques au niveau fenêtre (MAE, RMSE, R² pour V et A) ; puis agrégation **par vidéo** via la médiane des prédictions des fenêtres d'une même vidéo (`groupby('vid')`), comparée au label vidéo (`first`). La médiane est robuste aux fenêtres aberrantes.
>
> **`train_subject_cv`** : `GroupKFold(n_splits=5)` → 5 partitions où chaque vidéo entière est soit en train soit en test (jamais coupée). Pour chaque fold : `StandardScaler` fitté sur le train uniquement, entraînement, prédiction, métriques. On **moyenne les métriques sur les 5 folds** → estimation bien moins bruitée qu'un split unique (v1). Les prédictions concaténées servent aux figures.

In [6]:
def evaluate_window_and_video(y_true, y_pred, vids_test):
    res = {
        'win_MAE_V':  mean_absolute_error(y_true[:, 0], y_pred[:, 0]),
        'win_MAE_A':  mean_absolute_error(y_true[:, 1], y_pred[:, 1]),
        'win_RMSE_V': np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0])),
        'win_RMSE_A': np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1])),
        'win_R2_V':   r2_score(y_true[:, 0], y_pred[:, 0]),
        'win_R2_A':   r2_score(y_true[:, 1], y_pred[:, 1]),
    }
    df = pd.DataFrame({'vid': vids_test,
                       'V_true': y_true[:, 0], 'A_true': y_true[:, 1],
                       'V_pred': y_pred[:, 0], 'A_pred': y_pred[:, 1]})
    g = df.groupby('vid').agg({'V_true': 'first', 'A_true': 'first',
                               'V_pred': 'median', 'A_pred': 'median'})
    res.update({
        'vid_MAE_V':  mean_absolute_error(g['V_true'], g['V_pred']),
        'vid_MAE_A':  mean_absolute_error(g['A_true'], g['A_pred']),
        'vid_RMSE_V': np.sqrt(mean_squared_error(g['V_true'], g['V_pred'])),
        'vid_RMSE_A': np.sqrt(mean_squared_error(g['A_true'], g['A_pred'])),
    })
    return res


def train_subject_cv(subject_id, model_factory, model_name='', n_splits=5):
    X, y, vids = extract_subject_features(subject_id)
    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []
    for tr, te in gkf.split(X, y, groups=vids):
        scaler = StandardScaler().fit(X[tr])
        model = model_factory().fit(scaler.transform(X[tr]), y[tr])
        y_pred = model.predict(scaler.transform(X[te]))
        fold_metrics.append(evaluate_window_and_video(y[te], y_pred, vids[te]))
    mean_metrics = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
    return {'subject': subject_id, 'model': model_name, **mean_metrics}

> **Explication technique — modèles & exécution**
>
> - `RandomForest` : 200 arbres, nativement multi-sortie (V,A). `GradientBoost` : enveloppé dans `MultiOutputRegressor` (1 modèle/sortie), profondeur 4, lr 0.05. `random_state=42` → reproductible.
> - Double boucle sujets × modèles, `train_subject_cv` → DataFrame `df_results` (1 ligne / sujet / modèle, métriques déjà moyennées sur 5 folds).

In [ ]:
MODEL_FACTORIES = {
    'RandomForest':  lambda: RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    'GradientBoost': lambda: MultiOutputRegressor(
        GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42)),
}

results = []
for sid in tqdm(SUBJECT_LIST, desc='Subjects (window CV)'):
    for name, factory in MODEL_FACTORIES.items():
        results.append(train_subject_cv(sid, factory, name))

df_results = pd.DataFrame(results)
df_results.head()

Subjects (window CV):   0%|          | 0/32 [00:00<?, ?it/s]

> **Explication technique — agrégation 32 sujets**
>
> Moyenne ± écart-type de chaque métrique sur les 32 sujets, par modèle. Les R² window-level resteront négatifs : un R²<0 signifie « pire que prédire la moyenne » — attendu ici car la cible est constante par vidéo et le modèle ne voit pas les vidéos de test (cf. §4).

In [ ]:
metric_cols = [c for c in df_results.columns if c.startswith(('win_', 'vid_'))]
agg = df_results.groupby('model')[metric_cols].agg(['mean', 'std'])
print('=== Window-level (GroupKFold 5) — moyenne sur 32 sujets ===')
print(agg.round(3))

## 4. Régression au niveau vidéo — Leave-One-Video-Out (le cadrage honnête)

Sur DEAP il y a **1 seul label V/A par vidéo de 60 s**. La vraie taille du problème par sujet est donc **40 points**, pas ~19 000 fenêtres (qui ne sont que des copies bruitées de 40 cibles). Ici on **agrège les fenêtres par vidéo** (moyenne des 85 features) → 40 lignes/sujet, puis **Leave-One-Video-Out** (40 folds : on prédit chaque vidéo en l'ayant exclue de l'entraînement). Le R² est alors calculé sur 40 points réellement indépendants.

> **Explication technique — dataset par vidéo + LOVO**
>
> - `aggregate_by_video` : `groupby(vid)` → moyenne des features sur toutes les fenêtres de la vidéo (signal lissé, bruit fenêtre réduit) ; la cible vidéo = `first` (constante par vidéo).
> - `LeaveOneGroupOut` avec `groups = video_id` ⇒ 40 folds : train sur 39 vidéos, test sur 1. `StandardScaler` re-fitté à chaque fold.
> - On compare 3 modèles adaptés au régime *petit n / grande dimension* (40 échantillons, 85 features) :
>   - `Dummy` (prédit la moyenne) → **référence R²=0** ;
>   - `Ridge` (linéaire régularisé, robuste en haute dimension) ;
>   - `RandomForest` (non linéaire).
> - Métriques sur les 40 prédictions hold-out : MAE/RMSE/R² pour V et A. C'est la métrique qui reflète l'usage réel.

In [ ]:
def aggregate_by_video(X, y, vids):
    df = pd.DataFrame(X, columns=ALL_FEATURE_NAMES)
    df['vid'] = vids
    Xv = df.groupby('vid')[ALL_FEATURE_NAMES].mean().values
    yv = pd.DataFrame({'vid': vids, 'V': y[:, 0], 'A': y[:, 1]}) \
            .groupby('vid').first()[['V', 'A']].values
    groups = np.sort(np.unique(vids))
    return Xv.astype(np.float32), yv.astype(np.float32), groups


LOVO_FACTORIES = {
    'Dummy':        lambda: DummyRegressor(strategy='mean'),
    'Ridge':        lambda: Ridge(alpha=10.0),
    'RandomForest': lambda: RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42),
}

def lovo_subject(subject_id):
    X, y, vids = extract_subject_features(subject_id)
    Xv, yv, groups = aggregate_by_video(X, y, vids)
    logo = LeaveOneGroupOut()
    rows = []
    for name, factory in LOVO_FACTORIES.items():
        preds = np.zeros_like(yv)
        for tr, te in logo.split(Xv, yv, groups=groups):
            scaler = StandardScaler().fit(Xv[tr])
            model = factory().fit(scaler.transform(Xv[tr]), yv[tr])
            preds[te] = model.predict(scaler.transform(Xv[te]))
        rows.append({
            'subject': subject_id, 'model': name,
            'MAE_V':  mean_absolute_error(yv[:, 0], preds[:, 0]),
            'MAE_A':  mean_absolute_error(yv[:, 1], preds[:, 1]),
            'RMSE_V': np.sqrt(mean_squared_error(yv[:, 0], preds[:, 0])),
            'RMSE_A': np.sqrt(mean_squared_error(yv[:, 1], preds[:, 1])),
            'R2_V':   r2_score(yv[:, 0], preds[:, 0]),
            'R2_A':   r2_score(yv[:, 1], preds[:, 1]),
        })
    return rows


lovo_results = []
for sid in tqdm(SUBJECT_LIST, desc='Subjects (LOVO video)'):
    lovo_results.extend(lovo_subject(sid))
df_lovo = pd.DataFrame(lovo_results)
df_lovo.head(6)

> **Explication technique — lecture des résultats LOVO**
>
> Moyenne ± std sur 32 sujets, par modèle. Lecture attendue : le **MAE** chute nettement vs window-level (la moyenne par vidéo lisse le bruit), mais le **R² reste souvent ≤ 0** même pour Ridge → preuve quantitative que, sur DEAP subject-specific, l'EEG+périph **ne prédit pas linéairement** la V/A continue à partir de 40 vidéos. Comparer chaque modèle au `Dummy` : s'il ne bat pas `Dummy` en R², il n'extrait aucun signal généralisable. C'est un résultat scientifique valide (et l'argument pour passer en classification high/low ou en cross-subject — cf. §6).

In [ ]:
agg_lovo = df_lovo.groupby('model')[['MAE_V','MAE_A','RMSE_V','RMSE_A','R2_V','R2_A']] \
                  .agg(['mean','std']).round(3)
print('=== Leave-One-Video-Out (40 folds) — moyenne sur 32 sujets ===')
print(agg_lovo)

best_lovo = df_lovo.groupby('model')['MAE_V'].mean().idxmin()
beats_dummy = (df_lovo[df_lovo.model==best_lovo].set_index('subject')['R2_V'] >
               df_lovo[df_lovo.model=='Dummy'].set_index('subject')['R2_V']).mean()
print(f'\nMeilleur modele (MAE_V) : {best_lovo}')
print(f'Sujets ou {best_lovo} bat Dummy en R2_V : {beats_dummy*100:.0f}%')

## 5. Visualisations

> **Explication technique — MAE par sujet : window vs vidéo (LOVO)**
>
> Histogrammes de la MAE par sujet. On superpose le window-level (meilleur modèle §3) et le LOVO vidéo (meilleur modèle §4) pour V et A. Le décalage des distributions vers la gauche en LOVO illustre concrètement le gain de l'agrégation par vidéo.

In [ ]:
best_win = df_results.groupby('model')['vid_MAE_V'].mean().idxmin()
win = df_results[df_results.model == best_win]
lov = df_lovo[df_lovo.model == best_lovo]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, dim in zip(axes, ['V', 'A']):
    ax.hist(win[f'vid_MAE_{dim}'], bins=15, alpha=0.55,
            label=f'window->video ({best_win})', color='steelblue')
    ax.hist(lov[f'MAE_{dim}'], bins=15, alpha=0.55,
            label=f'LOVO video ({best_lovo})', color='coral')
    ax.set_xlabel(f'MAE {dim}'); ax.set_ylabel('# sujets')
    ax.set_title(f'MAE {"Valence" if dim=="V" else "Arousal"} par sujet'); ax.legend()
plt.suptitle('Window-level vs Leave-One-Video-Out', y=1.03)
plt.tight_layout(); plt.show()

> **Explication technique — importance des features (RandomForest, LOVO)**
>
> Pour chaque sujet : RF entraîné sur le dataset **agrégé par vidéo** (40×85), récupération de `feature_importances_` (réduction d'impureté), moyenne sur 32 sujets. Barres colorées EEG (bleu) vs périphérique (corail). La part totale captée par les périphériques quantifie l'apport réel de GSR/BVP/EMG vs EEG pour la V/A.

In [ ]:
imp_list = []
for sid in tqdm(SUBJECT_LIST, desc='Feat importance (video-level)'):
    X, y, vids = extract_subject_features(sid)
    Xv, yv, _ = aggregate_by_video(X, y, vids)
    scaler = StandardScaler().fit(Xv)
    rf = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)
    rf.fit(scaler.transform(Xv), yv)
    imp_list.append(rf.feature_importances_)

imp = np.array(imp_list).mean(axis=0)
df_imp = pd.DataFrame({'feature': ALL_FEATURE_NAMES, 'importance': imp}) \
            .sort_values('importance', ascending=False)

top_n = 25
fig, ax = plt.subplots(figsize=(8, 7))
top = df_imp.head(top_n).iloc[::-1]
colors = ['coral' if f in PERIPH_FEATURE_NAMES else 'steelblue' for f in top['feature']]
ax.barh(top['feature'], top['importance'], color=colors)
ax.set_xlabel('Importance moyenne (32 sujets, dataset video-level)')
ax.set_title(f'Top {top_n} features — bleu=EEG, corail=peripherique')
plt.tight_layout(); plt.show()

periph_share = df_imp[df_imp.feature.isin(PERIPH_FEATURE_NAMES)]['importance'].sum()
print(f"Part d'importance captee par les peripheriques : {periph_share*100:.1f}%")

> **Explication technique — vrai vs prédit (LOVO, sujet exemple)**
>
> Pour le sujet 01 et le meilleur modèle LOVO : nuage des 40 prédictions hold-out vs vérités terrain (1 point = 1 vidéo). La diagonale rouge = prédiction parfaite. Si les points forment un nuage horizontal autour de la moyenne ⇒ R²≈0 (le modèle prédit ~la moyenne) — visualisation directe de la conclusion du §4.

In [ ]:
sid = '01'
X, y, vids = extract_subject_features(sid)
Xv, yv, groups = aggregate_by_video(X, y, vids)
logo = LeaveOneGroupOut()
preds = np.zeros_like(yv)
for tr, te in logo.split(Xv, yv, groups=groups):
    sc = StandardScaler().fit(Xv[tr])
    preds[te] = LOVO_FACTORIES[best_lovo]().fit(sc.transform(Xv[tr]), yv[tr]) \
                                            .predict(sc.transform(Xv[te]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, d, lab in zip(axes, [0, 1], ['Valence', 'Arousal']):
    ax.scatter(yv[:, d], preds[:, d], alpha=0.6, s=40)
    ax.plot([1, 9], [1, 9], 'r--', lw=1)
    ax.set_xlim(1, 9); ax.set_ylim(1, 9)
    ax.set_xlabel(f'{lab} vrai'); ax.set_ylabel(f'{lab} predit')
    ax.set_title(f'{lab} — R2={r2_score(yv[:, d], preds[:, d]):.2f}')
fig.suptitle(f'Sujet {sid} — LOVO {best_lovo} (40 videos)', y=1.02)
plt.tight_layout(); plt.show()

## 6. Notes & next steps

**Corrections appliquées en v2**
- Baseline pré-stimulus (3 s) exclue du fenêtrage → plus de fenêtres mal étiquetées.
- Welch réellement moyenné (`nperseg=128`) → PSD à variance réduite.
- Window-level en **GroupKFold 5** (estimations stables) au lieu d'un split unique.
- Ajout du **Leave-One-Video-Out au niveau vidéo** = cadrage honnête (40 points/sujet) avec baseline `Dummy` pour ancrer le R².
- Warnings non masqués ; cache versionné `cache_full_v2`.

**Ce que montrent les résultats**
- Le **R² window-level reste négatif** : structurel sur DEAP (cible constante/vidéo, vidéos de test non vues), ce n'est pas un défaut d'implémentation.
- En LOVO le **MAE s'améliore** (agrégation = débruitage) mais le **R² reste proche de 0** : à 40 vidéos/sujet, l'EEG+périph ne prédit pas de façon fiable la V/A *continue* en subject-specific.

**Étapes suivantes recommandées**
1. **Classification high/low** (seuil 5 sur l'échelle 1–9) — protocole DEAP standard où ces features fonctionnent réellement.
2. **Cross-subject / pooled** avec `LeaveOneSubjectOut` → bien plus de vidéos d'entraînement.
3. Features physiologiques plus parlantes : **HRV** (détection de pics sur BVP), **breathing rate** (pics Resp), asymétrie alpha frontale, cohérence inter-canaux.
4. Calibration per-subject des sorties (z-score puis dénormalisation) pour absorber l'usage hétérogène de l'échelle 1–9.
5. Coupler les prédictions au scoring de zones optimales (`test.ipynb`) pour la sortie finale du système.

## 7. Sauvegarde & reprise (persistance)

Objectif : **ne plus repartir de zéro** à chaque lancement (l'entraînement a pris plusieurs jours).

Trois choses sont rendues persistantes sur le disque :

| Élément | Où | Statut |
|---|---|---|
| **Données extraites** (features EEG+périph) | `cache_full_v2/sXX_full_v2.npz` | ✅ déjà automatique via `extract_subject_features` (cache `.npz`) |
| **Résultats des modèles** (`df_results`, `df_lovo`, `df_imp`) | `artifacts/*.pkl` + `.csv` | ➕ cellule « SAUVEGARDE » ci-dessous |
| **Modèles entraînés** (sklearn) | `artifacts/models/*.joblib` | ➕ cellule « MODÈLES » ci-dessous |

**Mode d'emploi**
1. **Maintenant** (kernel déjà rempli après vos jours de calcul) : exécutez la cellule **« SAUVEGARDE IMMÉDIATE »** → elle écrit `df_results / df_lovo / df_imp` sur disque **sans rien recalculer**.
2. (Optionnel) Exécutez la cellule **« ENTRAÎNER & SAUVER LES MODÈLES »** pour persister des modèles finaux réutilisables en inférence (réentraînement unique, puis chargement instantané).
3. **Sessions futures** : exécutez les cellules 1→2 (imports + config + définitions de fonctions, instantanées grâce au cache `.npz`), puis la cellule **« REPRISE »** — et **sautez les cellules d'entraînement lourdes** (§3 et §4).

> Ces cellules sont **additives** : elles n'altèrent ni le notebook ni les résultats existants.

> **Explication technique — helpers de persistance**
>
> - `ARTIFACTS_DIR` : dossier dédié `notebooks/artifacts` (résultats) + `artifacts/models` (modèles), créés si absents.
> - `save_df` / `load_df` : DataFrame → **pickle** (préserve dtypes, aucune dépendance) **+ CSV** lisible à l'œil. `load_df` renvoie `None` si le fichier n'existe pas (→ permet un chargement conditionnel sans erreur).
> - `save_obj` / `load_obj` : objets sklearn (modèles, `StandardScaler`) via **`joblib`** (efficace pour les gros tableaux numpy internes des forêts).
> - Tout est encapsulé pour être ré-exécutable sans risque (idempotent).

In [ ]:
import os, joblib, pandas as pd

ARTIFACTS_DIR = r"E:\COURS ECOLE\COURS PGE 4\AI CLINIC\notebooks\artifacts"
MODELS_DIR    = os.path.join(ARTIFACTS_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)


def save_df(df, name):
    """DataFrame -> pickle (+ csv lisible). name sans extension."""
    p = os.path.join(ARTIFACTS_DIR, name)
    df.to_pickle(p + ".pkl")
    df.to_csv(p + ".csv", index=False)
    print(f"  [df] {name}: {df.shape} -> {p}.pkl (+ .csv)")


def load_df(name):
    """Renvoie le DataFrame s'il existe sur disque, sinon None."""
    p = os.path.join(ARTIFACTS_DIR, name + ".pkl")
    if os.path.exists(p):
        df = pd.read_pickle(p)
        print(f"  [df] {name}: charge {df.shape} depuis disque")
        return df
    print(f"  [df] {name}: absent du disque (None)")
    return None


def save_obj(obj, name):
    """Objet sklearn (modele, scaler...) -> joblib. name sans extension."""
    p = os.path.join(MODELS_DIR, name + ".joblib")
    joblib.dump(obj, p, compress=3)
    return p


def load_obj(name):
    """Renvoie l'objet joblib s'il existe, sinon None."""
    p = os.path.join(MODELS_DIR, name + ".joblib")
    return joblib.load(p) if os.path.exists(p) else None


print("Helpers de persistance prets. Dossiers :")
print(" -", ARTIFACTS_DIR)
print(" -", MODELS_DIR)

> **Explication technique — SAUVEGARDE IMMÉDIATE (à lancer maintenant)**
>
> Capture **sans aucun recalcul** ce qui est déjà en mémoire après vos jours de calcul. Chaque variable est sauvegardée seulement si elle existe (`if 'x' in globals()`), donc la cellule ne plante pas même si une partie n'a pas été exécutée. Après ça, vos résultats survivent à un redémarrage du kernel.

In [ ]:
saved = []
if 'df_results' in globals():
    save_df(df_results, 'df_results'); saved.append('df_results')
if 'df_lovo' in globals():
    save_df(df_lovo, 'df_lovo'); saved.append('df_lovo')
if 'df_imp' in globals():
    save_df(df_imp, 'df_imp'); saved.append('df_imp')

if saved:
    print(f"\nOK -> resultats persistes : {', '.join(saved)}")
else:
    print("Rien en memoire a sauvegarder. Lancez d'abord les cellules d'entrainement (§3/§4),")
    print("OU utilisez la cellule REPRISE pour recharger depuis le disque.")

> **Explication technique — ENTRAÎNER & SAUVER LES MODÈLES (optionnel, 1 seule fois)**
>
> Persiste des **modèles finaux réutilisables** pour l'inférence :
> - **Window-level** : pour chaque sujet et chaque modèle de `MODEL_FACTORIES`, un `StandardScaler` + le régresseur sont entraînés sur **toutes** les fenêtres du sujet, puis sauvegardés ensemble (`{'scaler','model'}`).
> - **Vidéo-level (LOVO)** : pour chaque sujet, `Ridge` et `RandomForest` sont entraînés sur le dataset agrégé par vidéo (40×85).
> - `force=False` → si le `.joblib` existe déjà, on **saute** (reprise propre, on ne réentraîne pas inutilement).
>
> ⚠️ C'est un réentraînement (potentiellement long la 1re fois) mais **unique** : ensuite `load_obj(...)` recharge le modèle en quelques ms. À ne lancer que si vous voulez les objets-modèles (les *métriques*, elles, sont déjà sauvées par la cellule précédente).

In [ ]:
from sklearn.preprocessing import StandardScaler

def train_and_save_final_models(force=False):
    for sid in tqdm(SUBJECT_LIST, desc='Save window-level models'):
        X, y, vids = extract_subject_features(sid)
        for mname, factory in MODEL_FACTORIES.items():
            tag = f"win_{mname}_s{sid}"
            if not force and load_obj(tag) is not None:
                continue
            sc = StandardScaler().fit(X)
            mdl = factory().fit(sc.transform(X), y)
            save_obj({'scaler': sc, 'model': mdl,
                      'features': ALL_FEATURE_NAMES, 'level': 'window'}, tag)

    for sid in tqdm(SUBJECT_LIST, desc='Save video-level models'):
        X, y, vids = extract_subject_features(sid)
        Xv, yv, _ = aggregate_by_video(X, y, vids)
        for mname in ('Ridge', 'RandomForest'):
            tag = f"vid_{mname}_s{sid}"
            if not force and load_obj(tag) is not None:
                continue
            sc = StandardScaler().fit(Xv)
            mdl = LOVO_FACTORIES[mname]().fit(sc.transform(Xv), yv)
            save_obj({'scaler': sc, 'model': mdl,
                      'features': ALL_FEATURE_NAMES, 'level': 'video'}, tag)
    print(f"\nModeles sauves dans : {MODELS_DIR}")

# Decommentez pour lancer (reentrainement unique) :
# train_and_save_final_models(force=False)
print("Appelez train_and_save_final_models() pour persister les modeles finaux.")

> **Explication technique — inférence depuis un modèle sauvé**
>
> `predict_subject(sid, level, model_name)` recharge le bundle `{scaler, model}` du disque (aucun entraînement) et prédit `[V, A]`. Sert à réutiliser un modèle entraîné sans relancer tout le pipeline.

In [ ]:
def predict_subject(sid, level='video', model_name='RandomForest', X=None):
    """Charge un modele sauve et predit [V,A]. level: 'window' | 'video'."""
    tag = f"{'win' if level=='window' else 'vid'}_{model_name}_s{sid}"
    bundle = load_obj(tag)
    if bundle is None:
        raise FileNotFoundError(f"Modele '{tag}' absent. Lancez train_and_save_final_models().")
    if X is None:
        Xf, y, vids = extract_subject_features(sid)
        X = Xf if level == 'window' else aggregate_by_video(Xf, y, vids)[0]
    Xs = bundle['scaler'].transform(X)
    return bundle['model'].predict(Xs)

print("predict_subject() pret (ex: predict_subject('01','video','Ridge')).")

> **Explication technique — REPRISE (à lancer en début de session future)**
>
> Recharge `df_results`, `df_lovo`, `df_imp` depuis `artifacts/*.pkl` **sans réentraîner**. À exécuter après les cellules 1→2 (imports + config + définitions, instantanées via le cache `.npz`) pour **sauter les cellules lourdes §3 et §4**. Si un fichier manque, la variable reste `None` et un message indique quoi relancer.

In [ ]:
df_results = load_df('df_results')
df_lovo    = load_df('df_lovo')
df_imp     = load_df('df_imp')

if df_results is not None:
    print("\n=== Reprise OK — apercu window-level ===")
    cols = [c for c in df_results.columns if c.startswith(('win_','vid_'))]
    print(df_results.groupby('model')[cols].mean().round(3))
if df_lovo is not None:
    print("\n=== Apercu LOVO video-level ===")
    print(df_lovo.groupby('model')[['MAE_V','MAE_A','R2_V','R2_A']].mean().round(3))
if df_results is None and df_lovo is None:
    print("Aucun resultat sur disque. Lancez les cellules d'entrainement (§3/§4)")
    print("puis la cellule 'SAUVEGARDE IMMEDIATE'.")